# Fase 3 — Fine-Tuning IndoSBERT (Google Colab)

**Jalankan di: Runtime → Change runtime type → GPU (T4)**

Alur:
1. Install dependency
2. Upload `data/train/pairs.jsonl` dari laptop
3. Fine-tune `firqaaa/indo-sentence-bert-base` dengan `MultipleNegativesRankingLoss` + 4 hard-negative
4. Unduh model hasil (`indosbert-legal-ft.zip`) ke laptop
5. Di laptop: ekstrak ke `models/indosbert-legal-ft/`, lalu jalankan `python scripts/08b_rebuild_index_ft.py`

## 1. Install & Cek GPU

In [ ]:
!pip install -q sentence-transformers datasets "accelerate>=1.1.0"

import os, torch
# Kurangi fragmentasi VRAM — penting untuk MNRL dengan banyak sequence per batch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU  :", torch.cuda.get_device_name(0))
    print("VRAM :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## 2. Upload `pairs.jsonl`

Klik **Choose Files** dan pilih `data/train/pairs.jsonl` dari laptop.

In [ ]:
from google.colab import files
uploaded = files.upload()   # pilih: data/train/pairs.jsonl

import json
PAIRS_FILE = list(uploaded.keys())[0]
pairs = [json.loads(l) for l in open(PAIRS_FILE, encoding='utf-8') if l.strip()]
print(f"Pasangan latih: {len(pairs)}")
print("Contoh:", pairs[0]['query'], '->', pairs[0]['positive'][:60], '...')

## 3. Konfigurasi Hyperparameter

> **Catatan VRAM (MNRL):** tiap batch MNRL memproses `BATCH_SIZE × (2 + NUM_HARD_NEG)` sequence sekaligus.
> Dengan `BATCH_SIZE=8` dan `NUM_HARD_NEG=4`, itu **48 sequence** per forward pass — aman di T4 15 GB.
> `GRAD_ACCUM=4` menjaga effective batch size = 32 (8 × 4).

In [ ]:
BASE_MODEL   = "firqaaa/indo-sentence-bert-base"
OUTPUT_DIR   = "indosbert-legal-ft"
EPOCHS       = 3
BATCH_SIZE   = 8      # ↓ dari 32; MNRL efektif = BATCH × (2+NEG) = 48 seq/batch
GRAD_ACCUM   = 4      # effective batch = 8 × 4 = 32
LEARNING_RATE= 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
MAX_SEQ_LEN  = 256
NUM_HARD_NEG = 4
SEED         = 42

## 4. Siapkan Dataset

In [ ]:
from datasets import Dataset

rows = []
for p in pairs:
    negs = p.get('hard_negatives', [])
    if len(negs) < NUM_HARD_NEG:
        continue
    row = {'anchor': p['query'], 'positive': p['positive']}
    for i in range(NUM_HARD_NEG):
        row[f'negative_{i+1}'] = negs[i]
    rows.append(row)

train_dataset = Dataset.from_list(rows)
print(f"Dataset : {len(rows)} baris")
print(f"Kolom   : {train_dataset.column_names}")

## 5. Fine-Tuning

In [ ]:
import math, torch
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

torch.cuda.empty_cache()  # bebaskan VRAM sisa sebelum mulai

model = SentenceTransformer(BASE_MODEL)
model.max_seq_length = MAX_SEQ_LEN
print(f"Model   : {BASE_MODEL}  max_seq_len={MAX_SEQ_LEN}")

loss = MultipleNegativesRankingLoss(model)

# hitung warmup_steps manual (warmup_ratio deprecated di Transformers v5+)
steps_per_epoch = math.ceil(len(rows) / (BATCH_SIZE * GRAD_ACCUM))
total_steps     = steps_per_epoch * EPOCHS
warmup_steps    = math.ceil(total_steps * WARMUP_RATIO)
print(f"Steps   : {total_steps} total, {warmup_steps} warmup")
print(f"VRAM sebelum train: {torch.cuda.memory_allocated()/1e9:.2f} GB")

args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_steps=warmup_steps,
    weight_decay=WEIGHT_DECAY,
    fp16=torch.cuda.is_available(),   # fp16 di GPU → hemat ~50% VRAM
    bf16=False,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    logging_steps=5,
    save_strategy="no",
    report_to=[],
    seed=SEED,
)

trainer = SentenceTransformerTrainer(
    model=model, args=args, train_dataset=train_dataset, loss=loss
)
trainer.train()
print("\nTraining selesai!")

## 6. Simpan & Unduh Model

In [ ]:
import os, shutil
from google.colab import files

model.save(OUTPUT_DIR)
print("File model:", os.listdir(OUTPUT_DIR))

zip_path = shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
print(f"Zip: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)")
files.download(zip_path)

## 7. Langkah Selanjutnya di Laptop

Setelah `indosbert-legal-ft.zip` terunduh:

```powershell
# 1. Ekstrak ke models/indosbert-legal-ft/
Expand-Archive indosbert-legal-ft.zip -DestinationPath models\indosbert-legal-ft

# 2. Rebuild FAISS index dengan model fine-tuned
python scripts/08b_rebuild_index_ft.py

# 3. Evaluasi ablasi lengkap
python scripts/09_evaluate_ablation.py
```